In [5]:
!pip install nflreadpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import nflreadpy as nfl

In [16]:
# One season of weekly player stats (one row per player per game)
# Pull in 2023 data to accurately simulate PPR draft
# Pull in 2024 data for rookies and actual testing
weekly23 = nfl.load_player_stats(2023).to_pandas()
weekly24 = nfl.load_player_stats(2024).to_pandas()
weekly23 = weekly23[(weekly23["season_type"] == "REG") & (weekly23.player_id.notna())]
weekly24 = weekly24[(weekly24["season_type"] == "REG") & (weekly24.player_id.notna())]

In [17]:
weekly24.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_NYJ_SF,...,0,0,0,0,0,0,0,0,8.58,8.58
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_ARI_BUF,...,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_TEN_CHI,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_JAX_MIA,...,0,0,0,0,0,0,0,0,0.00,0.00
4,00-0026498,M.Stafford,Matthew Stafford,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_LA_DET,...,0,0,0,0,0,0,0,0,14.68,14.68


# Draft Simulation

Use 2023 data to simulate a 2024 ppr draft. Utilize 2024 data to in-fill rookies. Draft rules:
- Snake draft
- 14 teams
- Starters: QB, 2 RB, 2 WR, TE, 1 FLEX
- 7 bench spots

In [18]:
from draft import *

In [19]:
board = build_board(season_totals(weekly23))
board = board[board.player_id.notna()].drop_duplicates("player_id")

draft_log, rosters = simulate_draft(board, seed=SEED)
waiver = build_waiver(board, set(draft_log.player_id), season_totals(weekly24))

print(draft_log.head(20))
print(lineup_strength(draft_log))
print(len(waiver), "players on waivers;", int(waiver.no_prior_data.sum()), "with no 2023 data")

    pick  round  slot  team   player_id                 name position  \
0      1      1     1     1  00-0033280  Christian McCaffrey       RB   
1      2      1     2     2  00-0036973       Travis Etienne       RB   
2      3      1     3     3  00-0036358          CeeDee Lamb       WR   
3      4      1     4     4  00-0033040          Tyreek Hill       WR   
4      5      1     5     5  00-0037840       Kyren Williams       RB   
5      6      1     6     6  00-0036963    Amon-Ra St. Brown       WR   
6      7      1     7     7  00-0038120          Breece Hall       RB   
7      8      1     8     8  00-0030279         Keenan Allen       WR   
8      9      1     9     9  00-0039040        De'Von Achane       RB   
9     10      1    10    10  00-0036322     Justin Jefferson       WR   
10    11      1    11    11  00-0034844       Saquon Barkley       RB   
11    12      1    12    12  00-0031687       Raheem Mostert       RB   
12    13      1    13    13  00-0036554         Nic

In [20]:
rosters[10]

,pick,round,slot,team,player_id,name,position,board_rank,proj_pts,vorp
9,10,1,10,10,00-0036322,Justin Jefferson,WR,8,283.504762,96.618487
18,19,2,5,10,00-0031408,Mike Evans,WR,26,250.184127,63.297852
37,38,3,10,10,00-0033553,James Conner,RB,23,224.941176,66.541176
46,47,4,5,10,00-0034753,Mark Andrews,TE,39,196.905495,49.336264
65,66,5,10,10,00-0030035,Adam Thielen,WR,65,210.946032,24.059757
74,75,6,5,10,00-0032950,Carson Wentz,QB,75,293.616000,16.986000
93,94,7,10,10,00-0037228,Jaylen Warren,RB,69,178.209524,19.809524
102,103,8,5,10,00-0037247,George Pickens,WR,91,194.031746,7.145472
121,122,9,10,10,00-0033873,Patrick Mahomes,QB,104,276.684000,0.054000
130,131,10,5,10,00-0032775,Demarcus Robinson,WR,120,177.688889,-9.197386


# RL on one season

**RL Environment:**
- One agent that owns a team, plays against 13 bots
- Runs about 150 steps/second
- The agent is only learning from stats in earlier weeks, plus the 2023 prior and the public bye weeks.
- Bots are also setting their line-ups each week, constrained by same rules as the agent

**Actions:**
- Actions for a player are start/sit, add/drop, trade
- Agent is limited to 5 actions per week (add/drop is considered 2 actions)
- Add/drop would only consider top-5 players available for a position
- Trade is only accepted by a bot if it improves bot's projected line-up by given margin
- Agent is limited to 2 add/drops, 1 trade

Rewards:
- Win = +1, loss = -1, small term for point margin
- At the end of the season there is a bonus based on final rank.




In [1]:
!pip install gymnasium sb3-contrib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 9.0 MB/s eta 0:00:00


In [21]:
from season import*

In [22]:
draft_logs = [simulate_draft(board, seed=s)[0] for s in range(200)]   # 200 different drafts

In [23]:
data = SeasonData(board, weekly24)

/content/season.py:14: RuntimeWarning: invalid value encountered in cast
  self.pos = ids.map(pos_map).map(POS_IDX).to_numpy().astype(int)


IndexError: index -9223372036854775808 is out of bounds for axis 0 with size 4